# SEBI URL/PDF Ingestion Sandbox (AJAX Search)

This notebook uses the actual request pattern captured from DevTools:
- `POST https://www.sebi.gov.in/sebiweb/ajax/search/section-search.jsp`

Flow:
1. Build AJAX payload that mirrors SEBI's search request.
2. Fetch search results via POST.
3. Parse result links returned by SEBI.
4. Rerank result titles with RapidFuzz against the query.
5. If top score passes threshold, open detail page, resolve PDF URL, download PDF.




In [ ]:
!pip install rapidfuzz


In [ ]:
from __future__ import annotations
import base64
import json
import re
from datetime import datetime
from pathlib import Path
from urllib.parse import parse_qs, unquote, urlencode, urljoin, urlparse
import requests
from bs4 import BeautifulSoup


In [ ]:
# Config
SECTION_SEARCH_BASE = "https://www.sebi.gov.in/section-search.html"
AJAX_SEARCH_ENDPOINT = "https://www.sebi.gov.in/sebiweb/ajax/search/section-search.jsp"
QUERY_TITLE = "Master Circular for AIFs"
# If exact date is known, use DD-MM-YYYY. Else keep None.
EXACT_DATE = "07-05-2024"
# cURL-aligned defaults from your captured request.
SEARCH_CONTEXT = "-1"
AJAX_FVAL = ""
AJAX_TYPE_SEARCH = "5"
# RapidFuzz rerank threshold
RAPIDFUZZ_MIN_SCORE = 0.0
# Optional: paste exact payload captured from DevTools if needed.
# Example: {"next_value":"1", ...}
RAW_AJAX_PAYLOAD_OVERRIDE = None
OUTPUT_DIR = Path("downloads")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TIMEOUT = 20
MAX_RETRIES = 2


In [ ]:
def normalize_text(value: str) -> str:
    return re.sub(r"\s+", " ", value).strip().casefold()
def encode_searchval_for_ajax(query_title: str) -> str:
    import base64
    # cURL flow sends base64-encoded query; use '+' for spaces before encoding.
    plain = query_title.replace(" ", "+")
    return base64.b64encode(plain.encode("utf-8")).decode("utf-8")
def build_section_search_url(query_title: str, search_context: str = SEARCH_CONTEXT, exact_date: str | None = None) -> str:
    params = {
        "searchval": query_title,
        "searchcontext": search_context,
        "searchfromdate": exact_date or "",
        "searchtodate": exact_date or "",
    }
    return f"{SECTION_SEARCH_BASE}?{urlencode(params)}"
def reranked_final_result(
    query_title: str,
    results: list[dict],
    min_rapidfuzz_score: float = RAPIDFUZZ_MIN_SCORE,
) -> dict:
    """RapidFuzz rerank on result titles and return top accepted match or rejection reason."""
    if not results:
        return {
            "status": "no_results",
            "reason": "No SEBI search results to rerank.",
            "reranked_results": [],
            "selected_result": None,
        }
    query_n = normalize_text(query_title)
    if not query_n:
        return {
            "status": "invalid_query",
            "reason": "Query is empty after normalization.",
            "reranked_results": [],
            "selected_result": None,
        }
    try:
        from rapidfuzz import fuzz
    except ImportError:
        return {
            "status": "rapidfuzz_unavailable",
            "reason": "rapidfuzz is not installed. Install with: pip install rapidfuzz",
            "reranked_results": [],
            "selected_result": None,
        }
    reranked_results = []
    for row in results:
        title = row.get("title", "")
        title_n = normalize_text(title)
        if not title_n:
            continue
        score = float(fuzz.token_set_ratio(query_n, title_n))
        reranked_results.append({
            "score": score,
            "result": row,
        })
    if not reranked_results:
        return {
            "status": "no_valid_titles",
            "reason": "No result titles available for RapidFuzz scoring.",
            "reranked_results": [],
            "selected_result": None,
        }
    reranked_results.sort(key=lambda x: x["score"], reverse=True)
    for i, item in enumerate(reranked_results, start=1):
        item["rank"] = i
    top = reranked_results[0]
    if top["score"] < float(min_rapidfuzz_score):
        return {
            "status": "rejected_threshold",
            "reason": f"Top score below threshold. rapidfuzz={top['score']:.2f} (min={min_rapidfuzz_score})",
            "reranked_results": reranked_results,
            "selected_result": None,
        }
    return {
        "status": "matched",
        "reason": "Top reranked result passed threshold.",
        "reranked_results": reranked_results,
        "selected_result": top["result"],
    }


In [ ]:
def make_session() -> requests.Session:
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/18.6 Safari/605.1.15",
        "Accept": "*/*",
        "Accept-Language": "en-US,en;q=0.9",
        "Origin": "https://www.sebi.gov.in",
    })
    return session
def fetch_html(session: requests.Session, url: str, retries: int = MAX_RETRIES) -> tuple[int, str, str]:
    last_err = None
    for _ in range(retries + 1):
        try:
            resp = session.get(url, timeout=TIMEOUT, allow_redirects=True)
            resp.raise_for_status()
            return resp.status_code, str(resp.url), resp.text
        except requests.RequestException as exc:
            last_err = exc
    raise RuntimeError(f"Failed to fetch HTML from {url}: {last_err}")
def build_ajax_payload(
    query_title: str,
    exact_date: str | None = None,
    fval: str = AJAX_FVAL,
    type_search: str = AJAX_TYPE_SEARCH,
) -> dict:
    return {
        "next_value": "1",
        "searchval": encode_searchval_for_ajax(query_title),
        "fromDate": exact_date or "",
        "toDate": exact_date or "",
        "fval": fval,
        "typeSearch": str(type_search),
        "next": "s",
        "doDirect": "-1",
        "sortby": "1",
    }
def fetch_search_results_ajax(
    session: requests.Session,
    query_title: str,
    exact_date: str | None = None,
    search_context: str = SEARCH_CONTEXT,
    fval: str = AJAX_FVAL,
    type_search: str = AJAX_TYPE_SEARCH,
    payload_override: dict | None = None,
) -> tuple[int, str, str, dict]:
    payload = payload_override or build_ajax_payload(
        query_title=query_title,
        exact_date=exact_date,
        fval=fval,
        type_search=type_search,
    )
    referer_url = build_section_search_url(query_title, search_context=search_context, exact_date=exact_date)
    headers = {
        "Content-Type": "application/x-www-form-urlencoded",
        "Referer": referer_url,
        "Sec-Fetch-Site": "same-origin",
        "Sec-Fetch-Mode": "cors",
        "Sec-Fetch-Dest": "empty",
    }
    last_err = None
    for _ in range(MAX_RETRIES + 1):
        try:
            resp = session.post(AJAX_SEARCH_ENDPOINT, data=payload, headers=headers, timeout=TIMEOUT, allow_redirects=True)
            resp.raise_for_status()
            return resp.status_code, str(resp.url), resp.text, payload
        except requests.RequestException as exc:
            last_err = exc
    raise RuntimeError(f"Failed AJAX search POST to {AJAX_SEARCH_ENDPOINT}: {last_err}")


In [ ]:
def infer_reference_type(text: str) -> str:
    t = normalize_text(text)
    if "master circular" in t or "master-circular" in t:
        return "master-circulars"
    if "regulation" in t:
        return "regulations"
    if "order" in t:
        return "orders"
    if "circular" in t:
        return "circulars"
    return "unknown"
def extract_section_search_results(html: str, base_url: str = "https://www.sebi.gov.in/") -> list[dict]:
    soup = BeautifulSoup(html, "html.parser")
    rows: list[dict] = []
    seen = set()
    for a in soup.find_all("a", href=True):
        title = a.get_text(" ", strip=True)
        href = urljoin(base_url, a["href"].strip())
        href_l = href.lower()
        if not title or len(title) < 4:
            continue
        if href in seen:
            continue
        if "sebi.gov.in" not in href_l:
            continue
        if "javascript:" in href_l:
            continue
        if not any(k in href_l for k in ["/legal/", "/circulars/", "/orders/", "/regulations/"]):
            continue
        context_text = " ".join([
            title,
            a.parent.get_text(" ", strip=True) if a.parent else "",
            href,
        ])
        rows.append({
            "title": title,
            "detail_url": href,
            "reference_type": infer_reference_type(context_text),
        })
        seen.add(href)
    return rows
def filter_by_reference_type(rows: list[dict], reference_type: str) -> list[dict]:
    wanted = normalize_text(reference_type)
    return [r for r in rows if normalize_text(r.get("reference_type", "")) == wanted]


In [ ]:
def extract_circular_metadata(circular_html: str) -> dict:
    soup = BeautifulSoup(circular_html, "html.parser")
    page_title = ""
    h1 = soup.find("h1")
    if h1:
        page_title = h1.get_text(" ", strip=True)
    text_blob = soup.get_text(" ", strip=True)
    circ_match = re.search(r"Circular\s*No\.?\s*[:-]?\s*([A-Za-z0-9\-/().]+)", text_blob, re.IGNORECASE)
    date_match = re.search(r"(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+\d{1,2},\s+\d{4}", text_blob, re.IGNORECASE)
    return {
        "page_title": page_title,
        "circular_no": circ_match.group(1) if circ_match else None,
        "date": date_match.group(0) if date_match else None,
    }
def extract_pdf_url(circular_html: str, base_url: str) -> str | None:
    soup = BeautifulSoup(circular_html, "html.parser")
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if ".pdf" in href.lower():
            return urljoin(base_url, href)
    for tag, attr in (("iframe", "src"), ("embed", "src"), ("object", "data")):
        for node in soup.find_all(tag):
            value = (node.get(attr) or "").strip()
            if ".pdf" in value.lower():
                return urljoin(base_url, value)
    absolute = re.search(r"https?://[^\"'\s>]+\.pdf(?:\?[^\"'\s>]*)?", circular_html, re.IGNORECASE)
    if absolute:
        return absolute.group(0)
    relative = re.search(r"([\"'])([^\"']+\.pdf(?:\?[^\"']*)?)\1", circular_html, re.IGNORECASE)
    if relative:
        return urljoin(base_url, relative.group(2))
    return None


In [ ]:
def slugify_filename(text: str, default: str = "sebi-document") -> str:
    cleaned = re.sub(r"[^a-zA-Z0-9]+", "-", text).strip("-").lower()
    return cleaned or default
def normalize_pdf_candidate_url(url: str) -> str:
    # Unwrap SEBI viewer links like /web/?file=<actual_pdf_url>
    parsed = urlparse(url)
    query = parse_qs(parsed.query)
    if "file" in query and query["file"]:
        return unquote(query["file"][0])
    return url
def download_pdf(session: requests.Session, pdf_url: str, output_dir: Path, filename_hint: str) -> Path:
    candidates = [normalize_pdf_candidate_url(pdf_url)]
    if candidates[0] != pdf_url:
        candidates.append(pdf_url)
    last_error = None
    final_resp = None
    for candidate in candidates:
        try:
            resp = session.get(candidate, timeout=TIMEOUT, allow_redirects=True)
            resp.raise_for_status()
            if resp.content.startswith(b"%PDF"):
                final_resp = resp
                break
            last_error = RuntimeError(
                f"Response not raw PDF for URL={candidate}; Content-Type={resp.headers.get('Content-Type', '')}"
            )
        except requests.RequestException as exc:
            last_error = exc
    if final_resp is None:
        raise RuntimeError(f"Could not download valid PDF from candidates={candidates}. Last error={last_error}")
    parsed = urlparse(str(final_resp.url))
    basename = Path(parsed.path).name
    if basename.lower().endswith(".pdf") and basename != ".pdf":
        filename = basename
    else:
        filename = f"{slugify_filename(filename_hint)}.pdf"
    out_path = output_dir / filename
    if out_path.exists():
        stamp = datetime.utcnow().strftime("%Y%m%d%H%M%S")
        out_path = output_dir / f"{out_path.stem}-{stamp}.pdf"
    out_path.write_bytes(final_resp.content)
    return out_path


In [ ]:
def sebi_search_tool(
    query_title: str,
    exact_date: str | None = None,
    search_context: str = SEARCH_CONTEXT,
    fval: str = AJAX_FVAL,
    type_search: str = AJAX_TYPE_SEARCH,
    payload_override: dict | None = None,
    session: requests.Session | None = None,
) -> dict:
    """Single-entry search tool: build payload -> AJAX fetch -> parse/display SEBI results."""
    sess = session or make_session()
    search_status, search_source, search_html, payload = fetch_search_results_ajax(
        sess,
        query_title=query_title,
        exact_date=exact_date,
        search_context=search_context,
        fval=fval,
        type_search=type_search,
        payload_override=payload_override,
    )
    parsed_rows = extract_section_search_results(search_html)
    return {
        "query_title": query_title,
        "exact_date": exact_date,
        "search_context": search_context,
        "fval": fval,
        "type_search": str(type_search),
        "ajax_endpoint": AJAX_SEARCH_ENDPOINT,
        "ajax_payload": payload,
        "search_status_code": search_status,
        "search_source": search_source,
        "result_count": len(parsed_rows),
        "results": parsed_rows,
        "raw_search_html": search_html,
    }


In [ ]:
def run_pipeline(
    query_title: str,
    output_dir: Path,
    exact_date: str | None = None,
    search_context: str = SEARCH_CONTEXT,
    fval: str = AJAX_FVAL,
    type_search: str = AJAX_TYPE_SEARCH,
    payload_override: dict | None = None,
    min_rapidfuzz_score: float = RAPIDFUZZ_MIN_SCORE,
) -> dict:
    session = make_session()
    try:
        search_out = sebi_search_tool(
            query_title=query_title,
            exact_date=exact_date,
            search_context=search_context,
            fval=fval,
            type_search=type_search,
            payload_override=payload_override,
            session=session,
        )
    except Exception as exc:
        return {
            "status": "failed",
            "stage": "search_tool",
            "error": str(exc),
            "endpoint": AJAX_SEARCH_ENDPOINT,
        }
    rerank_out = reranked_final_result(
        query_title=query_title,
        results=search_out["results"],
        min_rapidfuzz_score=min_rapidfuzz_score,
    )
    if rerank_out["status"] != "matched":
        return {
            "status": "failed",
            "stage": "reranked_final_result",
            "error": rerank_out["reason"],
            "query_title": query_title,
            "search_status_code": search_out["search_status_code"],
            "result_count": search_out["result_count"],
            "rerank": rerank_out,
        }
    selected = rerank_out["selected_result"]
    try:
        _, detail_final_url, detail_html = fetch_html(session, selected["detail_url"])
    except Exception as exc:
        return {
            "status": "failed",
            "stage": "fetch_detail",
            "error": str(exc),
            "selected_result": selected,
            "rerank": rerank_out,
        }
    metadata = extract_circular_metadata(detail_html)
    pdf_url = extract_pdf_url(detail_html, detail_final_url)
    if not pdf_url:
        return {
            "status": "failed",
            "stage": "resolve_pdf",
            "error": "No PDF URL found on detail page",
            "selected_result": selected,
            "detail_url": detail_final_url,
            "metadata": metadata,
            "rerank": rerank_out,
        }
    try:
        downloaded_path = download_pdf(session, pdf_url, output_dir, selected["title"])
    except Exception as exc:
        return {
            "status": "failed",
            "stage": "download_pdf",
            "error": str(exc),
            "selected_result": selected,
            "pdf_url": pdf_url,
            "metadata": metadata,
            "rerank": rerank_out,
        }
    return {
        "status": "success",
        "query_title": query_title,
        "exact_date": exact_date,
        "search_context": search_context,
        "fval": fval,
        "type_search": str(type_search),
        "ajax_endpoint": search_out["ajax_endpoint"],
        "ajax_payload": search_out["ajax_payload"],
        "search_status_code": search_out["search_status_code"],
        "search_source": search_out["search_source"],
        "result_count": search_out["result_count"],
        "rerank": rerank_out,
        "selected_result": selected,
        "detail_url": detail_final_url,
        "metadata": metadata,
        "pdf_url": pdf_url,
        "downloaded_pdf_path": str(downloaded_path),
    }


## Function-Level Tests (Run in Order)




In [ ]:
# 1) make_session
session = make_session()
assert isinstance(session, requests.Session)
print("make_session: ok")


In [ ]:
# 2) build_ajax_payload
payload = RAW_AJAX_PAYLOAD_OVERRIDE or build_ajax_payload(
    query_title=QUERY_TITLE,
    exact_date=EXACT_DATE,
    fval=AJAX_FVAL,
    type_search=AJAX_TYPE_SEARCH,
)
print(json.dumps(payload, indent=2))
assert "searchval" in payload and isinstance(payload["searchval"], str)


In [ ]:
# 3) sebi_search_tool (single entry for search flow)
search_out = sebi_search_tool(
    query_title=QUERY_TITLE,
    exact_date=EXACT_DATE,
    search_context=SEARCH_CONTEXT,
    fval=AJAX_FVAL,
    type_search=AJAX_TYPE_SEARCH,
    payload_override=RAW_AJAX_PAYLOAD_OVERRIDE,
    session=session,
)
print("search_status=", search_out["search_status_code"])
print("search_source=", search_out["search_source"])
print("result_count=", search_out["result_count"])
assert search_out["search_status_code"] == 200


In [ ]:
search_html


In [ ]:
# 4) inspect results from tool (no local matching)
results = search_out["results"]
print("results_count=", len(results))
if results:
    print("sample=", results[0])
assert isinstance(results, list)


In [ ]:
# 5) reranked_final_result (RapidFuzz)
rerank_out = reranked_final_result(
    query_title=QUERY_TITLE,
    results=results,
    min_rapidfuzz_score=RAPIDFUZZ_MIN_SCORE,
)
print("rerank_status=", rerank_out["status"])
print("reason=", rerank_out["reason"])
print("top3=", rerank_out["reranked_results"][:3])
assert rerank_out["status"] in {"matched", "rejected_threshold", "no_results", "invalid_query", "rapidfuzz_unavailable", "no_valid_titles"}


In [ ]:
print(QUERY_TITLE)
print(results)


In [ ]:
# 6) use selected top result for downstream detail/pdf tests
assert rerank_out["status"] == "matched", f"No acceptable match: {rerank_out['reason']}"
selected_result = rerank_out["selected_result"]
print("selected_result=", selected_result)
assert selected_result is not None


In [ ]:
# 7) fetch_html(detail) for selected result
detail_status, detail_final_url, detail_html = fetch_html(session, selected_result["detail_url"])
print("detail_status=", detail_status)
print("detail_final_url=", detail_final_url)
assert detail_status == 200


In [ ]:
# 8) extract_circular_metadata
meta = extract_circular_metadata(detail_html)
print(json.dumps(meta, indent=2))
assert isinstance(meta, dict)


In [ ]:
# 9) extract_pdf_url
pdf_url = extract_pdf_url(detail_html, detail_final_url)
print("pdf_url=", pdf_url)
assert pdf_url is not None and ".pdf" in pdf_url.lower()


In [ ]:
# 10) download_pdf
downloaded_path = download_pdf(session, pdf_url, OUTPUT_DIR, selected_result["title"])
print("downloaded_path=", downloaded_path)
assert downloaded_path.exists()
assert downloaded_path.suffix.lower() == ".pdf"


## Integration Test




In [ ]:
result = run_pipeline(
    query_title=QUERY_TITLE,
    output_dir=OUTPUT_DIR,
    exact_date=EXACT_DATE,
    search_context=SEARCH_CONTEXT,
    fval=AJAX_FVAL,
    type_search=AJAX_TYPE_SEARCH,
    payload_override=RAW_AJAX_PAYLOAD_OVERRIDE,
    min_rapidfuzz_score=RAPIDFUZZ_MIN_SCORE,
)
print(json.dumps(result, indent=2))


## Notes
- This notebook follows SEBI AJAX POST search flow captured from browser logs.
- Results are reranked using `rapidfuzz` on title text via `reranked_final_result`.
- `reranked_final_result` applies thresholds and can reject weak matches.
- Install RapidFuzz if needed: `pip install rapidfuzz`.
- If needed, paste exact captured payload into `RAW_AJAX_PAYLOAD_OVERRIDE`.


